# Tip 4 — Volume / Order-Flow Features

Returns alone are symmetric noise; signed volume reveals conviction behind each move.

1. **Signed-volume momentum** — sums at horizons 3, 5, 10, 20 + cross-sectional rank per date
2. **Return–volume interaction** — dot product and correlation between `RET_k` and `SIGNED_VOLUME_k`: positive = price moved *with* flow (continuation), negative = price moved *against* flow (divergence/reversal)
3. **Volume spike** — z-score of recent absolute volume vs its own 20-day norm; a reversal on a spike is a different animal from one on quiet tape

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
sns.set()

import lightgbm as lgbm
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold

import tip4_features

### Load data

In [ ]:
X_train = pd.read_csv('../../data/X_train.csv', index_col='ROW_ID')
X_test  = pd.read_csv('../../data/X_test.csv',  index_col='ROW_ID')
y_train = pd.read_csv('../../data/y_train.csv', index_col='ROW_ID')
sample_submission = pd.read_csv('../../data/sample_submission.csv', index_col='ROW_ID')

RET_cols = [f'RET_{i}'           for i in range(1, 21)]
VOL_cols = [f'SIGNED_VOLUME_{i}' for i in range(1, 21)]

### Feature engineering

#### A. Benchmark features

In [ ]:
def add_benchmark_features(df):
    for i in [3, 5, 10, 15, 20]:
        df[f'AVERAGE_PERF_{i}'] = df[RET_cols[:i]].mean(axis=1)
        df[f'ALLOCATIONS_AVERAGE_PERF_{i}'] = df.groupby('TS')[f'AVERAGE_PERF_{i}'].transform('mean')
    df['STD_PERF_20'] = df[RET_cols].std(axis=1)
    df['ALLOCATIONS_STD_PERF_20'] = df.groupby('TS')['STD_PERF_20'].transform('mean')
    return df

X_train = add_benchmark_features(X_train)
X_test  = add_benchmark_features(X_test)

benchmark_features = (
    RET_cols + VOL_cols + ['MEDIAN_DAILY_TURNOVER']
    + [f'AVERAGE_PERF_{i}'             for i in [3, 5, 10, 15, 20]]
    + [f'ALLOCATIONS_AVERAGE_PERF_{i}' for i in [3, 5, 10, 15, 20]]
    + ['STD_PERF_20', 'ALLOCATIONS_STD_PERF_20']
)

#### B. Tip 4 — Volume / order-flow features

In [ ]:
X_train, tip4_feats = tip4_features.add_features(X_train, RET_cols, VOL_cols)
X_test,  _          = tip4_features.add_features(X_test,  RET_cols, VOL_cols)

print(f"Tip 4 features ({len(tip4_feats)}):", tip4_feats)

#### Sanity check — feature distributions

In [ ]:
X_train[tip4_feats].describe().T[['mean', 'std', 'min', 'max']]

### Build final feature list

In [ ]:
features = benchmark_features + tip4_feats
print(f"Total features: {len(features)}  "
      f"(benchmark={len(benchmark_features)}, tip4={len(tip4_feats)})")

### LightGBM — 8-fold cross-validation on dates

In [ ]:
lgbm_params = {
    "objective":     "mse",
    "metric":        "mse",
    "num_threads":   50,
    "seed":          42,
    "verbosity":     -1,
    "learning_rate": 1e-2,
    "max_depth":     3,
}
NUM_BOOST_ROUND = 500

train_dates = X_train['TS'].unique()
scores, models = [], []

splits = KFold(n_splits=8, shuffle=True, random_state=0).split(train_dates)

for fold, (tr_idx, val_idx) in enumerate(splits):
    tr_mask  = X_train['TS'].isin(train_dates[tr_idx])
    val_mask = X_train['TS'].isin(train_dates[val_idx])

    X_tr  = X_train.loc[tr_mask,  features].fillna(0)
    y_tr  = y_train.loc[tr_mask,  'target']
    X_val = X_train.loc[val_mask, features].fillna(0)
    y_val = y_train.loc[val_mask, 'target']

    model = lgbm.train(lgbm_params,
                       lgbm.Dataset(X_tr, label=y_tr.values),
                       num_boost_round=NUM_BOOST_ROUND)

    preds = model.predict(X_val.values, num_threads=lgbm_params['num_threads'])
    acc   = accuracy_score((y_val > 0).astype(int), (preds > 0).astype(int))

    models.append(model)
    scores.append(acc)
    print(f"Fold {fold+1} — Accuracy: {acc*100:.2f}%")

mean = np.mean(scores) * 100
std  = np.std(scores)  * 100
print(f"\nAccuracy: {mean:.2f}% ± {std:.2f}%  [{mean-std:.2f} ; {mean+std:.2f}]")

### Feature importance (top 30 by gain)

In [ ]:
importances = pd.DataFrame(
    [m.feature_importance(importance_type='gain') for m in models],
    columns=features
)
top30 = importances.mean().sort_values(ascending=False).head(30).index

plt.figure(figsize=(10, 9))
sns.barplot(data=importances[top30], orient='h',
            order=importances[top30].mean().sort_values(ascending=False).index)
plt.title("Top-30 features by mean gain (8-fold)")
plt.tight_layout()
plt.show()

### Final model + submission

In [ ]:
final_model = lgbm.train(
    lgbm_params,
    lgbm.Dataset(X_train[features].fillna(0), label=y_train['target'].values),
    num_boost_round=NUM_BOOST_ROUND
)

test_preds = final_model.predict(X_test[features].fillna(0).values)
submission = pd.DataFrame(
    (test_preds > 0).astype(int),
    index=sample_submission.index,
    columns=['TARGET']
)
submission.to_csv('preds_tip4_volume_orderflow.csv')
print("Saved preds_tip4_volume_orderflow.csv")
submission['TARGET'].value_counts()